In [ ]:
# Pass 1: Intall libraries
!pip install openai PyMuPDF --quiet


In [ ]:
# Pass 2: Librerías necesarias
from openai import OpenAI
import fitz  # PyMuPDF
from getpass import getpass
from google.colab import files

In [ ]:
# Pass 3: Load PFF TICKET
print("📎 Load your ticket in PDF")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]


In [ ]:
# Paso 4: Extract text of PDF
def extraer_texto_pdf(ruta_pdf):
    doc = fitz.open(ruta_pdf)
    texto = ""
    for pagina in doc:
        texto += pagina.get_text()
    doc.close()
    return texto

boleta_texto = extraer_texto_pdf(pdf_path)
boleta_texto

In [ ]:
# Paso 5: API key
api_key = getpass("🔐 Ingresa tu API key de OpenAI: ").strip()
client = OpenAI(api_key=api_key)

In [ ]:
# Paso 6: Prompt para extraer los datos clave de la boleta
prompt_extraccion = f"""
Eres un experto en análisis de boletas electrónicas del Perú.

A partir del siguiente texto extraído de un archivo PDF, quiero que identifiques y devuelvas únicamente los siguientes campos, como **un JSON plano** (sin anidaciones ni listas), donde cada campo sea una clave independiente:

- Fecha de emisión
- RUC del emisor
- Cantidad
- Unidad de medida
- Código
- Descripción
- Valor unitario
- Descuento
- Importe de venta
- IGV
- Importe total

Devuelve **solo** el bloque JSON, sin explicaciones ni texto adicional. No incluyas ninguna lista bajo nombres como "Detalle" o "Items". Cada valor debe estar directamente en el objeto raíz del JSON.

Ejemplo esperado:

{{
  "Fecha de emisión": "dd/mm/aaaa",
  "RUC del emisor": "12345678901",
  "Cantidad": "1.00",
  "Unidad de medida": "UNIDAD",
  ...
}}

Texto de la boleta extraído por OCR:

\"\"\"
{boleta_texto}
\"\"\"
"""

# Paso 7: Extraer los datos con GPT
respuesta_extraccion = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Eres un analista experto en boletas de venta electrónicas SUNAT."},
        {"role": "user", "content": prompt_extraccion}
    ],
    temperature=0
)

datos_extraidos = respuesta_extraccion.choices[0].message.content.strip()

# Mostrar los datos extraídos
print("📋 Datos extraídos de la boleta:\n")
print(datos_extraidos)

# Paso 8: Análisis estructurado con otro prompt
prompt_analisis = f"""
A partir de estos datos extraídos de una boleta electrónica:

{datos_extraidos}

1. Valida que los campos numéricos sean coherentes (valor unitario + IGV ≈ total).
2. Señala si hay errores o inconsistencias.
3. Describe el tipo de producto o servicio.
4. Evalúa si cumple con lo requerido por SUNAT para un CPE válido.
5. Genera insights tributarios o contables relevantes.
"""

respuesta_analisis = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Eres un auditor tributario SUNAT especializado en comprobantes electrónicos."},
        {"role": "user", "content": prompt_analisis}
    ],
    temperature=0
)

# Mostrar el análisis final
print("\n🧠 Análisis de la boleta:\n")
print(respuesta_analisis.choices[0].message.content.strip())


In [ ]:
datos_extraidos

In [ ]:
# Paso 7.1: Guardar datos extraídos como JSON
import json

# Intentar convertir el texto a JSON (maneja errores si el formato no es válido)
try:
    datos_dict = json.loads(datos_extraidos)
    with open("boleta_extraida.json", "w", encoding="utf-8") as f:
        json.dump(datos_dict, f, ensure_ascii=False, indent=2)
    print("✅ Los datos fueron guardados como 'boleta_extraida.json'")
except Exception as e:
    print("⚠️ Error al guardar el archivo JSON:", e)


In [ ]:
datos_extraidos

In [ ]:
datos_dict


In [ ]:
import pandas as pd

# Convertir JSON plano a DataFrame
df = pd.DataFrame([datos_dict])

# Mostrar
df


## ✅ Conclusion

The conversion of structured data from an electronic receipt in PDF format to flat JSON and then into a DataFrame is a highly efficient and scalable solution for automating accounting and tax processes.

By using language models like GPT, we achieved:

- 📄 **Extracting key information** from unstructured documents (free text or scanned).
- 🧠 **Standardizing required fields** using a robust prompt, avoiding nested structures.
- 📊 **Automatically transforming results** into a `DataFrame` ready for analysis, validation, or export.
- ⚙️ **Reducing human errors** and accelerating repetitive tasks like receipt audits or tax filings.

This approach combines **artificial intelligence + natural language processing + data automation**, creating real impact in areas such as accounting, compliance, tax education, and document management.
